# Treinamento Final — Gradient Boosting (Dataset Brasileiro)

**Modelo:** `HistGradientBoostingClassifier` (scikit-learn)  
**Dataset:** `dataset_phishing_brasileiro.csv`  
**Objetivo:** Treinar e salvar o modelo final para detecção de phishing em URLs brasileiras

### Pipeline
1. Monta Drive e carrega CSV
2. Extrai 37 features numéricas por URL
3. **Busca de hiperparâmetros** via `RandomizedSearchCV`
4. Treina modelo final com melhores params + `early_stopping`
5. Avalia (métricas, curvas, feature importance)
6. Salva modelo `.pkl` em `/MyDrive/phishing_gradientboosting/`

### Referência — Benchmark (708k URLs, kmack/Phishing_urls)
| Métrica | GBM Benchmark | Random Forest |
|---|---|---|
| F1-Score | 83.3% | 84.2% |
| AUC-ROC | 91.0% | 91.6% |
| MCC | 0.659 | 0.677 |
| Latência P50 | **3.36ms** | 45ms |
| Tempo treino | **24.8s** | 155.5s |

> GBM foi selecionado pelo melhor equilíbrio F1/latência (13x mais rápido que RF com performance próxima).

**Antes de rodar:**
1. Faça upload do `dataset_phishing_brasileiro.csv` no Google Drive
2. Ajuste `CSV_PATH` na célula de configuração se necessário

In [ ]:
# ============================================================
# 1. Instalação de dependências
# ============================================================
!pip install -q scikit-learn pandas numpy matplotlib seaborn tldextract joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import re
import warnings
import os
warnings.filterwarnings('ignore')

from urllib.parse import urlparse
import tldextract
import joblib

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
)
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, auc,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, log_loss, matthews_corrcoef,
    precision_recall_curve, average_precision_score
)

print('Imports OK.')

In [ ]:
# ============================================================
# 2. Montar Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
print('Drive montado com sucesso.')

In [ ]:
# ============================================================
# 3. Configuração
# ============================================================
CSV_PATH     = '/content/drive/MyDrive/dataset_phishing_brasileiro.csv'
OUTPUT_DIR   = '/content/drive/MyDrive/phishing_gradientboosting'

RANDOM_STATE = 42
TEST_SIZE    = 0.2
CV_FOLDS     = 5

# Número de combinações testadas na busca de hiperparâmetros
# Aumente para busca mais exaustiva (mais tempo); diminua se o dataset for pequeno
N_ITER_SEARCH = 30

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Pasta de saída: {OUTPUT_DIR}')

In [ ]:
# ============================================================
# 4. Carregar Dataset
# ============================================================
df = pd.read_csv(CSV_PATH)

print(f'Total de registros: {len(df):,}')
print(f'Colunas: {list(df.columns)}')
print(f'\nDistribuição de labels:')
print(df['label'].value_counts())
print(f'  Phishing:  {df["label"].sum():,} ({df["label"].mean():.1%})')
print(f'  Legítimo:  {(df["label"] == 0).sum():,} ({(df["label"] == 0).mean():.1%})')

if 'source' in df.columns:
    print(f'\nDistribuição por fonte:')
    print(df['source'].value_counts())

df.head(10)

In [ ]:
# ============================================================
# 5. Feature Engineering (Versão Final Turbo — Brasil)
#    - Separação marca vs ação com features de interação
#    - Selos de confiança: .gov.br, .edu.br, .org.br
#    - brand_is_domain + neutralização de action keywords
# ============================================================

def extract_features(url: str) -> dict:
    """Extrai features otimizadas para marcas brasileiras e instituições."""
    raw = url
    if not url.startswith(('http://', 'https://')):
        url = 'http://' + url

    try:
        parsed = urlparse(url)
        ext = tldextract.extract(url)
    except Exception:
        parsed = urlparse('http://invalid')
        ext = tldextract.extract('http://invalid')

    hostname  = (parsed.hostname or '').lower()
    path      = (parsed.path or '').lower()
    query     = (parsed.query or '').lower()
    domain    = ext.domain.lower()
    subdomain = ext.subdomain.lower()
    suffix    = ext.suffix.lower()

    features = {}

    # --- ESTRUTURA BÁSICA ---
    features['url_length']      = len(raw)
    features['hostname_length'] = len(hostname)
    features['path_length']     = len(path)
    features['query_length']    = len(query)
    features['path_depth']      = path.count('/') - 1 if path else 0
    features['num_params']      = query.count('=') if query else 0

    # --- CARACTERES ---
    features['num_dots']        = raw.count('.')
    features['num_hyphens']     = raw.count('-')
    features['num_underscores'] = raw.count('_')
    features['num_slashes']     = raw.count('/')
    features['num_ats']         = raw.count('@')
    features['num_equals']      = raw.count('=')
    features['num_digits']      = sum(c.isdigit() for c in raw)
    features['num_special']     = sum(not c.isalnum() and c not in './-_' for c in raw)

    # --- DIFERENCIAÇÃO LOCAL ---
    features['num_hyphens_domain'] = hostname.count('-')
    features['num_hyphens_path']   = path.count('-')
    features['num_dots_hostname']  = hostname.count('.')

    # --- SELOS DE CONFIANÇA ---
    features['is_gov_br']        = int(ext.suffix == 'gov.br')
    features['is_com_br']        = int(ext.suffix == 'com.br')
    features['is_edu_br']        = int(ext.suffix == 'edu.br')
    features['is_org_br']        = int(ext.suffix == 'org.br')

    features['subdomain_length'] = len(ext.subdomain)
    features['domain_length']    = len(ext.domain)
    features['tld_length']       = len(ext.suffix)
    features['num_subdomains']   = ext.subdomain.count('.') + 1 if ext.subdomain else 0

    # --- PROPORÇÕES E ENTROPIA ---
    features['digit_ratio']   = features['num_digits']  / max(len(raw), 1)
    features['special_ratio'] = features['num_special'] / max(len(raw), 1)
    if len(raw) > 0:
        prob = [raw.count(c) / len(raw) for c in set(raw)]
        features['entropy'] = -sum(p * np.log2(p) for p in prob if p > 0)
    else:
        features['entropy'] = 0

    # Entropia separada do domínio (forte discriminador segundo Tamal et al. 2024)
    dom_str = ext.domain + '.' + ext.suffix if ext.suffix else ext.domain
    if len(dom_str) > 0:
        prob_d = [dom_str.count(c) / len(dom_str) for c in set(dom_str)]
        features['entropy_domain'] = -sum(p * np.log2(p) for p in prob_d if p > 0)
    else:
        features['entropy_domain'] = 0

    # --- PADRÕES SUSPEITOS ---
    features['has_ip'] = int(bool(re.match(r'^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$', hostname)))
    features['has_at_sign']               = int('@' in raw)
    features['has_double_slash_redirect'] = int('//' in raw[8:])
    features['has_hex_encoding']          = int('%' in raw)
    features['has_https']                 = int(raw.startswith('https'))
    features['has_www']                   = int('www.' in hostname)
    features['has_prefix_suffix']         = int('-' in hostname)

    shorteners = ['bit.ly', 'goo.gl', 'shorte.st', 'go2l.ink', 'x.co', 'ow.ly', 't.co', 'tinyurl.com', 'tr.im', 'is.gd', 'cli.re']
    features['has_shortener'] = int(hostname in shorteners or any(hostname.endswith('.' + s) for s in shorteners))

    try:
        port = parsed.port
    except ValueError:
        port = None
    features['has_port'] = int(port is not None and port not in [80, 443])

    suspicious_ext = ['.exe', '.zip', '.rar', '.js', '.php', '.cgi', '.asp', '.aspx', '.scr', '.bat', '.cmd']
    features['has_suspicious_ext'] = int(any(path.endswith(e) for e in suspicious_ext))

    # --- NOVAS FEATURES (literatura: Tamal 2024, Trieuh2) ---
    # Punycode — ataques homógrafos com domínios internacionalizados (xn--)
    features['has_punycode'] = int('xn--' in hostname)

    # TLDs gratuitos/suspeitos muito usados em phishing
    suspicious_tlds = ['.tk', '.ml', '.ga', '.cf', '.gq', '.xyz', '.top', '.buzz', '.club', '.info', '.work', '.link']
    features['suspicious_tld'] = int(any(suffix.endswith(t.lstrip('.')) for t in suspicious_tlds))

    # TLD aparece no path (ex: evil.com/google.com/login)
    common_tlds = ['.com', '.org', '.net', '.gov', '.edu', '.com.br', '.org.br']
    features['tld_in_path'] = int(any(t in path for t in common_tlds))

    # Dígitos no domínio registrado (legítimos raramente têm)
    features['has_digits_in_domain'] = int(any(c.isdigit() for c in domain))

    # --- PALAVRAS-CHAVE (Separação Marca vs Ação) ---
    action_keywords = [
        'login', 'signin', 'verify', 'account', 'update', 'secure', 'confirm', 'password', 'suspend',
        'conta', 'acesso', 'seguro', 'atualizar', 'liberar', 'bloqueio', 'recadastro', 'atencao',
        'beneficio', 'auxilio', 'urgente'
    ]
    brand_keywords = [
        # Brasil
        'caixa', 'bradesco', 'itau', 'nubank', 'santander', 'bb', 'inter', 'picpay',
        # Global
        'google', 'microsoft', 'apple', 'amazon', 'openai', 'github',
        'netflix', 'facebook', 'whatsapp', 'instagram', 'twitter', 'linkedin',
        'paypal', 'mercadolivre', 'magazineluiza', 'americanas',
    ]

    is_brand = int(domain in brand_keywords)

    # Marca no domínio registrado = LEGÍTIMO (ex: itau.com.br → 1)
    features['brand_is_domain'] = is_brand

    # Marca no subdomínio ou hostname mas NÃO é o domínio = SUSPEITO
    features['brand_in_subdomain'] = int(
        any(kw in subdomain for kw in brand_keywords) or
        (any(kw in hostname for kw in brand_keywords) and not is_brand)
    )

    # Action keywords NEUTRALIZADAS: zeradas quando o domínio é marca legítima
    # itau.com.br/conta/acesso → 0 (é o banco real, "conta" e "acesso" são normais)
    # fake-itau.xyz/conta/acesso → 2 (domínio falso, "conta" e "acesso" são suspeitos)
    action_h = sum(kw in hostname for kw in action_keywords)
    action_p = sum(kw in path for kw in action_keywords)
    action_q = sum(kw in query for kw in action_keywords)

    features['action_kw_hostname'] = action_h * (1 - is_brand)
    features['action_kw_path']     = action_p * (1 - is_brand)
    features['action_kw_query']    = action_q * (1 - is_brand)

    # --- ANÁLISE DE TOKENS ---
    tokens = re.split(r'[.\-_/=?&]', raw)
    features['num_tokens']       = len(tokens)
    features['avg_token_length'] = np.mean([len(t) for t in tokens if t]) if tokens else 0
    features['max_token_length'] = max((len(t) for t in tokens if t), default=0)

    return features

print('Extraindo features das URLs...')
t0 = time.time()

url_col = 'url' if 'url' in df.columns else 'text'
feature_rows = df[url_col].apply(extract_features)
df_features  = pd.DataFrame(feature_rows.tolist())

print(f'Features extraídas em {time.time() - t0:.1f}s')
print(f'Shape: {df_features.shape} — {df_features.shape[1]} features')
df_features.head()

In [ ]:
# ============================================================
# 6. Pré-processamento e Split
# ============================================================
X = df_features.copy()
y = df['label'].astype(int)

X = X.replace([np.inf, -np.inf], np.nan)
nulos = X.isnull().sum().sum()
if nulos > 0:
    print(f'Valores nulos: {nulos} — preenchendo com mediana.')
    X = X.fillna(X.median())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,} | Features: {X_train.shape[1]}')
print(f'Train — Phishing: {y_train.sum():,} ({y_train.mean():.1%}) | Legítimo: {(~y_train.astype(bool)).sum():,}')
print(f'Test  — Phishing: {y_test.sum():,}  ({y_test.mean():.1%}) | Legítimo: {(~y_test.astype(bool)).sum():,}')

In [ ]:
# ============================================================
# 7. Busca de Hiperparâmetros (RandomizedSearchCV)
#    Features mudaram (51 features + correções) — refazer busca
#    max_iter aumentado: modelo anterior atingiu 300/300 sem convergir
# ============================================================
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

param_dist = {
    'max_iter':          [500, 1000, 1500, 2000],
    'learning_rate':     [0.01, 0.03, 0.05, 0.08, 0.1],
    'max_depth':         [4, 5, 6, 8, None],
    'max_leaf_nodes':    [31, 63, 127, 255],
    'min_samples_leaf':  [10, 20, 30, 50],
    'l2_regularization': [0.0, 0.01, 0.1, 0.5, 1.0],
}

base_estimator = HistGradientBoostingClassifier(
    early_stopping=True, validation_fraction=0.1,
    n_iter_no_change=20, scoring='f1', random_state=RANDOM_STATE
)

print(f'Executando RandomizedSearchCV ({N_ITER_SEARCH} combinações, 3-fold)...')
t_search = time.time()

search = RandomizedSearchCV(
    estimator=base_estimator, param_distributions=param_dist,
    n_iter=N_ITER_SEARCH, scoring='f1',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=-1, verbose=1, random_state=RANDOM_STATE, refit=False,
)
search.fit(X_train, y_train)

best_params = search.best_params_
print(f'\nBusca concluída em {time.time() - t_search:.1f}s')
print(f'Melhor F1 CV: {search.best_score_:.4f}')
print(f'Melhores params:')
for k, v in best_params.items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# 8. Treino do Modelo Final
#    Usa os melhores hiperparâmetros + early_stopping no conjunto completo
# ============================================================
print('Treinando modelo final com melhores hiperparâmetros...')
print(f'Params: {best_params}')

gbm = HistGradientBoostingClassifier(
    **best_params,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    scoring='f1',
    random_state=RANDOM_STATE
)

t_start = time.time()
gbm.fit(X_train, y_train)
train_time = time.time() - t_start

n_iters_used = gbm.n_iter_
max_configured = best_params.get('max_iter', 500)
print(f'\nTreino concluído em {train_time:.2f}s')
print(f'Iterações usadas: {n_iters_used} / {max_configured} (early stopping parou no momento certo)')

if n_iters_used >= max_configured:
    print('AVISO: atingiu o max_iter sem convergir — considere aumentar max_iter.')
else:
    print(f'Early stopping parou {max_configured - n_iters_used} iterações antes do limite — sem overfitting.')

In [ ]:
# ============================================================
# 9. Avaliação no Conjunto de Teste
# ============================================================
y_pred  = gbm.predict(X_test)
y_proba = gbm.predict_proba(X_test)[:, 1]

acc         = accuracy_score(y_test, y_pred)
prec        = precision_score(y_test, y_pred)
rec         = recall_score(y_test, y_pred)
f1          = f1_score(y_test, y_pred)
auc_roc     = roc_auc_score(y_test, y_proba)
ap          = average_precision_score(y_test, y_proba)
mcc         = matthews_corrcoef(y_test, y_pred)
logloss     = log_loss(y_test, y_proba)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
specificity = tn / (tn + fp)

latencies = []
for i in range(min(500, len(X_test))):
    sample = X_test.iloc[[i]]
    t0 = time.perf_counter()
    gbm.predict(sample)
    latencies.append((time.perf_counter() - t0) * 1000)
latencies = np.array(latencies)

# Referência do benchmark (708k URLs)
BENCH_F1      = 0.8329
BENCH_AUC_ROC = 0.9102
BENCH_MCC     = 0.6589

print('=' * 65)
print('RESULTADOS — GBM Final (Dataset Brasileiro)')
print('=' * 65)
print(f'{"Métrica":<20} {"Atual":>10}   {"Bench (708k)":>14}   {"Delta":>8}')
print('-' * 65)
print(f'{"Accuracy":<20} {acc:>10.4f}')
print(f'{"Precision":<20} {prec:>10.4f}')
print(f'{"Recall":<20} {rec:>10.4f}')
print(f'{"F1-Score":<20} {f1:>10.4f}   {BENCH_F1:>14.4f}   {f1 - BENCH_F1:>+8.4f}')
print(f'{"AUC-ROC":<20} {auc_roc:>10.4f}   {BENCH_AUC_ROC:>14.4f}   {auc_roc - BENCH_AUC_ROC:>+8.4f}')
print(f'{"MCC":<20} {mcc:>10.4f}   {BENCH_MCC:>14.4f}   {mcc - BENCH_MCC:>+8.4f}')
print(f'{"PR-AUC":<20} {ap:>10.4f}')
print(f'{"Log Loss":<20} {logloss:>10.4f}')
print(f'{"Specificity":<20} {specificity:>10.4f}')
print(f'{"Latência P50 (ms)":<20} {np.percentile(latencies, 50):>10.2f}')
print(f'{"Latência P95 (ms)":<20} {np.percentile(latencies, 95):>10.2f}')
print(f'{"Latência P99 (ms)":<20} {np.percentile(latencies, 99):>10.2f}')
print(f'{"Iterações usadas":<20} {gbm.n_iter_:>10}')
print(f'{"Tempo treino (s)":<20} {train_time:>10.2f}')
print('=' * 65)
print()
print(classification_report(y_test, y_pred, target_names=['Legítimo', 'Phishing']))

In [ ]:
# ============================================================
# 10. Cross-Validation (5-Fold) com os melhores hiperparâmetros
# ============================================================
print(f'Executando Cross-Validation ({CV_FOLDS}-Fold) com melhores params...')
print('(early_stopping desativado no CV para garantir comparação justa entre folds)\n')

# No CV, desativamos early_stopping para que todos os folds usem
# exatamente n_iters_used iterações (as que o modelo final convergiu)
cv_params = {k: v for k, v in best_params.items()}
cv_params['max_iter'] = gbm.n_iter_   # usa o número real de iterações do modelo final

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

cv_scores = cross_validate(
    HistGradientBoostingClassifier(**cv_params, random_state=RANDOM_STATE),
    X, y,
    cv=cv,
    scoring={'accuracy': 'accuracy', 'f1': 'f1', 'roc_auc': 'roc_auc', 'precision': 'precision', 'recall': 'recall'},
    n_jobs=-1
)

# Referência CV do benchmark
BENCH_CV_F1_MEAN  = 0.8323
BENCH_CV_F1_STD   = 0.0007
BENCH_CV_AUC_MEAN = 0.9094
BENCH_CV_AUC_STD  = 0.0008

print('Cross-Validation Results:')
print(f'  {"Métrica":<12} {"Atual (mean±std)":>22}   {"Bench (708k)":>22}')
print('  ' + '-' * 60)

cv_f1_mean  = cv_scores['test_f1'].mean()
cv_f1_std   = cv_scores['test_f1'].std()
cv_auc_mean = cv_scores['test_roc_auc'].mean()
cv_auc_std  = cv_scores['test_roc_auc'].std()

print(f'  {"accuracy":<12} {cv_scores["test_accuracy"].mean():.4f} ± {cv_scores["test_accuracy"].std():.4f}')
print(f'  {"f1":<12} {cv_f1_mean:.4f} ± {cv_f1_std:.4f}   {BENCH_CV_F1_MEAN:.4f} ± {BENCH_CV_F1_STD:.4f}')
print(f'  {"roc_auc":<12} {cv_auc_mean:.4f} ± {cv_auc_std:.4f}   {BENCH_CV_AUC_MEAN:.4f} ± {BENCH_CV_AUC_STD:.4f}')
print(f'  {"precision":<12} {cv_scores["test_precision"].mean():.4f} ± {cv_scores["test_precision"].std():.4f}')
print(f'  {"recall":<12} {cv_scores["test_recall"].mean():.4f} ± {cv_scores["test_recall"].std():.4f}')

print()
folds_f1 = [f'{v:.4f}' for v in cv_scores['test_f1']]
print(f'  F1 por fold: {folds_f1}')

if cv_f1_std < 0.01:
    print(f'\n  Modelo estável (std F1 = {cv_f1_std:.4f} < 0.01)')
else:
    print(f'\n  ATENÇÃO: variância alta entre folds (std F1 = {cv_f1_std:.4f}) — dataset pode ser pequeno ou desbalanceado.')

In [ ]:
# ============================================================
# 11. Visualizações
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Matriz de Confusão
ax = axes[0]
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Legítimo', 'Phishing'])
disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
ax.set_title('Matriz de Confusão', fontsize=13, fontweight='bold')

# Curva ROC
ax = axes[1]
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_val = auc(fpr, tpr)
ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {roc_val:.4f} (atual)')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random (AUC = 0.50)')
ax.fill_between(fpr, tpr, alpha=0.1, color='steelblue')
ax.set_xlabel('FPR')
ax.set_ylabel('TPR')
ax.set_title('Curva ROC', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

# Curva Precision-Recall
ax = axes[2]
prec_vals, rec_vals, _ = precision_recall_curve(y_test, y_proba)
ax.plot(rec_vals, prec_vals, color='darkorange', lw=2, label=f'AP = {ap:.4f}')
ax.fill_between(rec_vals, prec_vals, alpha=0.1, color='darkorange')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Curva Precision-Recall', fontsize=13, fontweight='bold')
ax.legend(loc='lower left')
ax.grid(True, alpha=0.3)

plt.suptitle('Gradient Boosting — Dataset Brasileiro', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, 'curvas_avaliacao.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Salvo: {plot_path}')

In [ ]:
# ============================================================
# 12. Importância das Features
# ============================================================
if hasattr(gbm, 'feature_importances_'):
    importances = gbm.feature_importances_
    feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(10, 8))
    feat_imp.tail(20).plot(kind='barh', ax=ax, color='steelblue', edgecolor='black', linewidth=0.5)
    ax.set_title('Top 20 Features — Gradient Boosting', fontsize=13, fontweight='bold')
    ax.set_xlabel('Importância')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()

    fi_path = os.path.join(OUTPUT_DIR, 'feature_importance.png')
    plt.savefig(fi_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Salvo: {fi_path}')

    df_fi = feat_imp.sort_values(ascending=False).reset_index()
    df_fi.columns = ['Feature', 'Importância']
    print('\nTop 10 features mais importantes:')
    print(df_fi.head(10).to_string(index=False))
else:
    print('feature_importances_ não disponível para este modelo.')

In [ ]:
# ============================================================
# 13. Top combinações encontradas na busca
# ============================================================
df_search = pd.DataFrame(search.cv_results_)
df_search = df_search.sort_values('rank_test_score')

cols_show = ['rank_test_score', 'mean_test_score', 'std_test_score',
             'param_max_iter', 'param_learning_rate', 'param_max_depth',
             'param_max_leaf_nodes', 'param_min_samples_leaf', 'param_l2_regularization']
print('Top 10 combinações (por F1):')
display(df_search[cols_show].head(10).round(4))

# Salvar resultados da busca
search_path = os.path.join(OUTPUT_DIR, 'hyperparameter_search_results.csv')
df_search[cols_show].to_csv(search_path, index=False)
print(f'\nResultados da busca salvos: {search_path}')

In [ ]:
# ============================================================
# 14. Salvar Modelo e Resultados no Drive
# ============================================================

# Modelo
model_path = os.path.join(OUTPUT_DIR, 'gbm_phishing_brasileiro.pkl')
joblib.dump(gbm, model_path)
print(f'Modelo salvo: {model_path}')

# Lista de features (necessária para inferência futura)
feature_names = list(X.columns)
features_path = os.path.join(OUTPUT_DIR, 'feature_names.pkl')
joblib.dump(feature_names, features_path)
print(f'Feature names salvo: {features_path}')

# Resultados
results_dict = {
    'Modelo':            ['Gradient Boosting'],
    'Dataset':           ['dataset_phishing_brasileiro'],
    'Accuracy':          [round(acc, 4)],
    'Precision':         [round(prec, 4)],
    'Recall':            [round(rec, 4)],
    'F1-Score':          [round(f1, 4)],
    'AUC-ROC':           [round(auc_roc, 4)],
    'PR-AUC':            [round(ap, 4)],
    'MCC':               [round(mcc, 4)],
    'Log Loss':          [round(logloss, 4)],
    'Specificity':       [round(specificity, 4)],
    'CV_F1_mean':        [round(cv_f1_mean, 4)],
    'CV_F1_std':         [round(cv_f1_std, 4)],
    'CV_AUC_mean':       [round(cv_auc_mean, 4)],
    'CV_AUC_std':        [round(cv_auc_std, 4)],
    'Latencia_P50_ms':   [round(np.percentile(latencies, 50), 2)],
    'Latencia_P95_ms':   [round(np.percentile(latencies, 95), 2)],
    'Latencia_P99_ms':   [round(np.percentile(latencies, 99), 2)],
    'n_iters_used':      [gbm.n_iter_],
    'Tempo_Treino_s':    [round(train_time, 2)],
    'N_train':           [len(X_train)],
    'N_test':            [len(X_test)],
    'best_params':       [str(best_params)],
}
df_results = pd.DataFrame(results_dict)
results_path = os.path.join(OUTPUT_DIR, 'resultados_gbm_brasileiro.csv')
df_results.to_csv(results_path, index=False)
print(f'Resultados salvos: {results_path}')

print('\n' + '='*65)
print('Arquivos salvos em:', OUTPUT_DIR)
print('  gbm_phishing_brasileiro.pkl        — modelo treinado')
print('  feature_names.pkl                  — lista de features (37)')
print('  resultados_gbm_brasileiro.csv      — métricas completas')
print('  hyperparameter_search_results.csv  — top combinações da busca')
print('  curvas_avaliacao.png               — ROC / PR / confusão')
print('  feature_importance.png             — importância das features')
print('='*65)

In [ ]:
# ============================================================
# 10b. Avaliação Específica (Dataset Brasileiro)
#      Como o dataset é 99% global, precisamos ver como ele se sai no 'brasil'
# ============================================================

if 'region' in df.columns:
    # Pegamos os índices do conjunto de teste que são da região 'brasil'
    br_indices = df.loc[X_test.index][df['region'].str.contains('brasil|sul', na=False)].index
    
    if len(br_indices) > 0:
        X_test_br = X_test.loc[br_indices]
        y_test_br = y_test.loc[br_indices]
        
        y_pred_br  = gbm.predict(X_test_br)
        y_proba_br = gbm.predict_proba(X_test_br)[:, 1]
        
        print('=' * 65)
        print('AVALIAÇÃO — APENAS URLS BRASILEIRAS (Subconjunto do Teste)')
        print('=' * 65)
        print(classification_report(y_test_br, y_pred_br, target_names=['Legítimo', 'Phishing']))
        
        cm_br = confusion_matrix(y_test_br, y_pred_br)
        print(f'Matriz de Confusão (Brasil):\n{cm_br}')
        
        # Analisar Falsos Positivos (Legítimos classificados como Phishing)
        fps = X_test_br[(y_test_br == 0) & (y_pred_br == 1)]
        if len(fps) > 0:
            print(f'\nTotal de Falsos Positivos no Brasil: {len(fps)}')
            print('Exemplos de URLs legítimas brasileiras marcadas como Phishing:')
            for idx in fps.head(10).index:
                print(f"  - {df.loc[idx, 'url']}")
    else:
        print('Nenhum dado brasileiro encontrado no conjunto de teste.')
else:
    print('Coluna "region" não encontrada no dataframe original.')

In [ ]:
# ============================================================
# 15. Exemplo de Inferência com o Modelo Salvo
# ============================================================
modelo_carregado   = joblib.load(model_path)
features_carregadas = joblib.load(features_path)

def predict_url(url: str, threshold: float = 0.5) -> dict:
    feats   = extract_features(url)
    X_input = pd.DataFrame([feats])[features_carregadas]
    X_input = X_input.replace([np.inf, -np.inf], np.nan).fillna(0)

    # Probabilidade da classe 1 (Phishing)
    proba_phishing = modelo_carregado.predict_proba(X_input)[0][1]

    # Decisão com base no limiar
    is_phishing = proba_phishing >= threshold

    # Confiança: quanto o modelo tem certeza do resultado
    # Se phishing: confiança = proba_phishing (ex: 0.96 → 96%)
    # Se legítimo: confiança = 1 - proba_phishing (ex: 0.03 → 97%)
    confianca = proba_phishing if is_phishing else (1 - proba_phishing)

    return {
        'url': url,
        'predicao': 'PHISHING' if is_phishing else 'LEGITIMO',
        'confianca': f'{confianca:.2%}',
        'proba_phishing': proba_phishing
    }

urls_teste = [
    # Legítimas
    'https://www.google.com.br',
    'https://www.uol.com.br',
    'https://www.globo.com',
    'https://www.bb.com.br',
    'https://www.caixa.gov.br',
    'https://www.mercadolivre.com.br',
    'https://www.amazon.com.br',
    'https://www.pudim.com.br',
    'https://www.gov.br',
    'https://www.receita.fazenda.gov.br',
    'https://presidencia.gov.br',
    'https://bradesco.com.br',
    'https://itau.com.br/conta/acesso',

    # Suspeitas / Phishing
    'http://192.168.1.1/login?account=verify&password=update',
    'http://bradesco-conta-bloqueada.xyz/login/verify/account',
    'http://itau-seguranca-conta.ml/login?token=abc123',
    'http://verificacao-itau-seguranca.com',
    'https://bradesco-atendimento-cliente.net',
    'http://caixa-auxilio-emergencial.info',
    'http://correios-rastreio-objeto.xyz',
    'https://portal-da-transparencia-vagas.com',
    'http://suporte-tecnico-microsoft.com.br.net',
    'http://recadastro-anatel.com.br.link'
]

LIMIAR = 0.5

print(f'Exemplos de predição (Threshold: {LIMIAR}):')
print('-' * 90)
for url in urls_teste:
    r = predict_url(url, threshold=LIMIAR)
    print(f"  [{r['predicao']:8s}] confiança: {r['confianca']:6s}  {r['url']}")